# Read LSSTCamSources in all bands (with time and field)

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-11
- **Last update:** 2026-07-11

## Goal

This notebook is the **offline companion** to
`01b_FindLSSTCamSourcesInAllbands_keeptimeandfield.ipynb`. It does **not**
touch the Butler: it simply reads back the parquet files written by
notebook 01b into `DIR_DATA_IN` (`./data_FindLSSTCamSources_01b`) and
reproduces the same plots:

- `objectstats_band_<band>.parquet` : one row per stable star (object),
  with `n_visits, ra, dec, flux_mean, flux_median, flux_mad, mag_median,
  sigmaF_over_F_phot, sigmaF_over_F_meas, mmag_meas, mmag_phot, ddf, band`.
- `lcdeviation_band_<band>.parquet` : one row per detection of a retained
  object, with `object_id, mjd, dmmag, band, ddf` (flux deviation from the
  object's own median flux, in mmag).

Plots reproduced (same section numbering as notebook 01b, section 9):
- 9.1 boxplot of `mmag_meas` per band (no outliers)
- 9.2 boxplot of `mmag_meas` per band (with outliers)
- 9.3 `mmag_meas` vs. median magnitude, per band
- 9.4 light curves: flux deviation from the per-object median (mmag) vs. MJD,
  one panel per band, color = band, marker = DDF

**Note:** the parquet files produced at USDF are not copied automatically
into this local directory; point `DIR_DATA_IN` below to wherever you have
synced/copied the `data_FindLSSTCamSources_01b/` folder (or run this
notebook directly at USDF, next to notebook 01b).

## 1. Imports

In [ ]:
import glob
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.time import Time

In [ ]:
# view all contents of pandas tables
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found \u2192 interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found \u2192 %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Le logging est configure et fonctionne dans le notebook !")

## 3. Configuration

**Edit only this cell** to point to the directory containing the parquet
files written by notebook 01b (`DIR_DATA_OUT` there). These constants are
only used for plot titles / labels here (the actual selection was already
applied in notebook 01b).

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────
NB_TAG = "ReadLSSTCamSourcesInAllbands_02b"

# ── Input: data written by notebook 01b ────────────────────────────────
DIR_DATA_IN = "./data_FindLSSTCamSources_01b"

# ── Output figures ──────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# All LSST bands, in the standard display order
BANDS = ["u", "g", "r", "i", "z", "y"]
BANDS_CMAP = {"u": "Purples", "g": "Greens", "r": "Reds", "i": "YlOrBr", "z": "pink_r", "y": "bone_r"}
BANDS_COLOR = {
    "u": "blueviolet",
    "g": "limegreen",
    "r": "red",
    "i": "darkorange",
    "z": "chocolate",
    "y": "saddlebrown",
}

# Marker style per DDF (used in the mmag-deviation-vs-MJD light-curve plots),
# same convention as notebook 01b
DDF_MARKERS = {
    "COSMOS": "o",
    "XMM-LSS": "s",
    "ECDFS": "^",
    "ELAIS-S1": "D",
    "EDFS": "v",
}
DDF_MARKER_DEFAULT = "x"  # fallback for any DDF not listed above

unique_ddf = list(DDF_MARKERS.keys())

# -- Magnitude window used in notebook 01b (for plot titles only) -----------
MAG_MIN = 17.0
MAG_MAX = 19.5

# -- Minimum number of visits per band used in notebook 01b (for plot titles)
MIN_VISITS_PER_BAND = {"u": 20, "g": 50, "r": 50, "i": 50, "z": 50, "y": 20}

log.info(f"Reading per-band parquet files from '{DIR_DATA_IN}'")

## 4. Helper functions

In [ ]:
# ── savefig: PDF + PNG ───────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
def add_top_date_axis(ax, mjd_min, mjd_max, n_ticks=6):
    """Add a secondary x-axis on top of *ax* showing calendar dates
    (YYYY-MM-DD) at evenly spaced MJD positions between mjd_min/mjd_max.
    """
    ticks_mjd = np.linspace(mjd_min, mjd_max, n_ticks)
    ticks_date = Time(ticks_mjd, format="mjd").to_value("iso", subfmt="date")
    secax = ax.secondary_xaxis("top")
    secax.set_xticks(ticks_mjd)
    secax.set_xticklabels(ticks_date, rotation=45, ha="left")
    secax.set_xlabel("Date")
    return secax

## 5. Read the per-band parquet files

### 5.1  Object-level statistics (`objectstats_band_<band>.parquet`)

In [ ]:
all_band_results = {}

for band in BANDS:
    path = os.path.join(DIR_DATA_IN, f"objectstats_band_{band}.parquet")
    if not os.path.exists(path):
        log.warning(f"Band '{band}': file not found ({path}), skipping")
        continue
    df_band = pd.read_parquet(path)
    all_band_results[band] = df_band
    log.info(f"Band '{band}': {len(df_band)} objects read from {path}")

if not all_band_results:
    raise FileNotFoundError(
        f"No objectstats_band_*.parquet files found in '{DIR_DATA_IN}'. "
        "Check DIR_DATA_IN in the configuration cell above."
    )

### 5.2  Per-detection light-curve deviations (`lcdeviation_band_<band>.parquet`)

In [ ]:
all_band_lc_results = {}

for band in BANDS:
    path = os.path.join(DIR_DATA_IN, f"lcdeviation_band_{band}.parquet")
    if not os.path.exists(path):
        log.warning(f"Band '{band}': file not found ({path}), skipping")
        continue
    df_band_lc = pd.read_parquet(path)
    all_band_lc_results[band] = df_band_lc
    log.info(f"Band '{band}': {len(df_band_lc)} detections read from {path}")

if not all_band_lc_results:
    log.warning(
        f"No lcdeviation_band_*.parquet files found in '{DIR_DATA_IN}'. "
        "Section 9.4 (light curves vs MJD) will be skipped."
    )

## 6. Combine all bands and inspect the results

In [ ]:
df_all = pd.concat(
    [df for df in all_band_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total objects across all bands/DDFs: {len(df_all)}")

df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

### 6b. Combine per-detection light-curve deviations across bands

In [ ]:
if all_band_lc_results:
    df_lc_all = pd.concat(
        [df for df in all_band_lc_results.values() if len(df) > 0],
        ignore_index=True,
    )
    log.info(f"Total per-detection light-curve deviations across all bands/DDFs: {len(df_lc_all)}")
else:
    df_lc_all = pd.DataFrame()

df_lc_all.head()

## 7. Plots (reproducing notebook 01b, section 9)

### 7.1  Plot: relative photometric scatter (mmag) per band \u2014 box plot, no outliers

Boxplot of `mmag_meas` (measured scatter, using `psfFlux`) grouped by band,
in the standard LSST band order `u, g, r, i, z, y`. We expect the largest
scatter in **u** and **y**.

In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(data, tick_labels=band_order, showfliers=False)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()

savefig(fig, "mmag_scatter_allsrc_perband_noout")

plt.show()

### 7.2  Plot: relative photometric scatter (mmag) per band \u2014 with outliers

In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
flier_props_71 = dict(
    marker="o",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="gray",
    markeredgewidth=1.0,
    alpha=0.6,
)
ax.boxplot(data, tick_labels=band_order, showfliers=True, flierprops=flier_props_71)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.0, 200.0)
plt.tight_layout()

savefig(fig, "mmag_scatter_allsrc_perband_without")

plt.show()

### 7.3  Plot: relative photometric scatter (mmag) vs magnitude

In [ ]:
# Cross-check: measured scatter vs. photon-noise-only expectation, per band
fig, ax = plt.subplots(figsize=(8, 6))
for b in band_order:
    sub = df_all.loc[df_all["band"] == b]
    ax.scatter(sub["mag_median"], sub["mmag_meas"], s=10, color=BANDS_COLOR[b], alpha=0.4, label=b)
ax.set_xlabel("median magnitude")
ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")
ax.set_yscale("log")
ax.legend(markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig(fig, "mmag_scatter_allsrc_allbands")
plt.show()

### 7.4  Light curves: flux deviation from the per-object median (mmag) vs MJD, per band

For every retained object and every one of its visits, we plot the deviation
of that visit's `psfFlux` from the object's own median flux (in mmag) against
the visit MJD, for **all objects at once**, one panel per band:

- **color** encodes the **band** (using `BANDS_COLOR`)
- **marker** encodes the **DDF** (using `DDF_MARKERS`)
- the **bottom** x-axis is MJD, the **top** x-axis shows the calendar date
  (`YYYY-MM-DD`)

In [ ]:
if df_lc_all.empty:
    log.warning("No light-curve deviation data available, skipping section 7.4")
else:
    # configuration of the figure
    band_order_lc = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_lc_all["band"].unique()]
    n_bands_lc = len(band_order_lc)
    ncols_lc = 2
    nrows_lc = int(np.ceil(n_bands_lc / ncols_lc))

    mjd_min_all = df_lc_all["mjd"].min()
    mjd_max_all = df_lc_all["mjd"].max()

    YAXIS_MAX_MAG = 500.0

    # create the figure
    fig, axes = plt.subplots(nrows_lc, ncols_lc, figsize=(16, 3.2 * nrows_lc), sharex=True)
    axes = np.atleast_1d(axes).ravel()

    # loop on bands
    for i, band in enumerate(band_order_lc):
        ax = axes[i]
        sub_band = df_lc_all.loc[df_lc_all["band"] == band]

        # loop on ddf
        for ddf_name in unique_ddf:
            sub = sub_band.loc[sub_band["ddf"] == ddf_name]
            if len(sub) == 0:
                continue
            marker = DDF_MARKERS.get(ddf_name, DDF_MARKER_DEFAULT)
            ax.scatter(
                sub["mjd"],
                sub["dmmag"],
                s=8,
                alpha=0.35,
                color=BANDS_COLOR[band],
                marker=marker,
                linewidths=0,
                label=ddf_name,
            )

        ax.axhline(0.0, color="k", lw=0.8, alpha=0.5)

        # clip y-range to the 0.5-99.5 percentile of this band to stay readable
        band_vals = sub_band["dmmag"].dropna().to_numpy()
        if len(band_vals) > 0:
            y_lo, y_hi = np.nanpercentile(band_vals, [0.5, 99.5])
            pad = 0.15 * (y_hi - y_lo) if y_hi > y_lo else 1.0
            ax.set_ylim(y_lo - pad, y_hi + pad)

        ax.set_ylabel(r"$\Delta$mmag")
        ax.set_title(f"band {band!r}")
        ax.grid(True, alpha=0.3)

        if i // ncols_lc == 0:
            add_top_date_axis(ax, mjd_min_all, mjd_max_all)
        if i // ncols_lc == nrows_lc - 1:
            ax.set_xlabel("MJD")
        if i == 0:
            ax.legend(markerscale=2.5, fontsize=8, ncol=2, title="DDF", loc="upper right")

        ax.set_ylim(-YAXIS_MAX_MAG, YAXIS_MAX_MAG)

    for j in range(n_bands_lc, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        "Per-object flux deviation from its median (mmag) vs time \u2014 color: band, marker: DDF",
        y=1.02,
    )
    plt.tight_layout()
    savefig(fig, "mmag_deviation_vs_mjd_perband")
    plt.show()